# `code/pipeline/i73_gender_screening.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
I-73 여성 파트너와 여성 창업자 — 호의(favoritism) vs 선별(screening) 판별 (K-3 본추정)

[왜] STAGE2_REPORT §5 K-3. JF(Cook–Marx–Yimfor, Funding Black Startups) 공식의 성별·글로벌판:
자연실험이 아니라 ① 매칭 사실(동류교배) 확립 → ② 출구성과로 경쟁 해석 판별.
호의(taste)라면 여성 파트너가 고른 여성 창업 딜은 낮은 기준으로 통과 → 성과 열위.
선별(정보 우위)이라면 과소평가된 풀에서 더 잘 고름 → 성과 동등 이상.
[설계]
  단위   = 파트너 귀속 딜 (investment_partners: round×investor×partner), 그랜트·부채 제외
  측정층 = fp = 파트너 성별(people.gender ∈ {male,female}); ff = 기업에 여성 창업자 ≥1
           (founder 직함 jobs × gender; 판정 가능 = 성별 관측 창업자 ≥1)
  H1 동류교배: ff ~ fp, 투자사×연도 셀 demean (FWL), 투자사 군집 부트 500. 2010–2023-10.
  H2 판별: 결과 = 출구(인수·IPO, 라운드 후 72개월; 라운드 2010–2017-10)
           및 후속 라운드(36개월; 라운드 2010–2020-10).
           gap_ff = (여성 파트너 − 남성 파트너) 성과 격차, ff=1 딜 내부, 투자사×연도 demean;
           gap_mf = 동일 계산, ff=0 딜. 판별자 = gap_ff − gap_mf (공동 투자사 부트).
  조절   = 미국 vs 비미국 (동류교배의 글로벌 이질성 — K-3 의 기여축).
[사전 예측] (결과 전, 2026-09-03)
  P1 동류교배: fp 딜의 ff 비중이 셀 내 +2~+8pp 높음, CI 0 배제 (기저 ff ~10–15% 가정).
  P2 판별: JF 의 인종 결과 유추 — 선별 방향. gap_ff ≥ 0, 판별자 CI 가 큰 음수(−10pp 이하) 배제.
  P3 출구·후속 두 결과변수의 부호 일치.
  P4 동류교배는 비미국에서 ≥ 미국 (여성 파트너 희소 시장에서 매칭 프리미엄 큼) — 탐색적.
[Kill] (사전 등록)
  K1 관측성: 파트너 성별 커버리지 < 60% 또는 창업자 성별 판정가능 딜 < 40% 또는
     (fp × ff 판정가능) 딜 < 2,000 → KILL.
  K2 동류교배 부재: H1 CI 0 포함 → 판별할 매칭 자체가 없음 — KILL.
  K3 판별력: gap_ff CI 폭이 출구·후속 모두 > 20pp → 메커니즘 판별 불능 — KILL.
```


In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from emit_contract import emit, qci  # noqa: E402
from gates import CTX  # noqa: E402

rng = np.random.default_rng(42)
NB = 500
EQ_EXCL = {"grant", "debt_financing", "post_ipo_debt", "post_ipo_equity", "non_equity_assistance"}

people = CTX.people[["uuid", "gender"]]
g_map = people[people["gender"].isin(["male", "female"])].set_index("uuid")["gender"]

rounds = CTX.rounds.dropna(subset=["announced_on", "org_uuid"]).copy()
rounds = rounds[~rounds["investment_type"].isin(EQ_EXCL)]
rounds["dt"] = pd.to_datetime(rounds["announced_on"], errors="coerce")
rounds = rounds.dropna(subset=["dt"])
rounds = rounds[(rounds["dt"] >= "2010-01-01") & (rounds["dt"] <= "2023-10-31")]

# 창업자 성별 → 기업 ff
jobs = CTX.jobs
fj = jobs[jobs["title"].fillna("").str.lower().str.contains("founder", regex=False)][
    ["person_uuid", "org_uuid"]].dropna()
fj["fg"] = fj["person_uuid"].map(g_map)
fj = fj[fj["fg"].notna()]
org_ff = (fj["fg"] == "female").groupby(fj["org_uuid"]).max()  # 여성 창업자 ≥1

# 파트너 귀속 딜
pt = CTX.partners.dropna(subset=["funding_round_uuid", "investor_uuid", "partner_uuid"])
d = pt.merge(rounds[["uuid", "org_uuid", "dt", "country_code"]],
             left_on="funding_round_uuid", right_on="uuid")
d["pg"] = d["partner_uuid"].map(g_map)
cov_pg = float(d["pg"].notna().mean())
d["ff"] = d["org_uuid"].map(org_ff)
cov_ff = float(d["ff"].notna().mean())
d = d[d["pg"].notna() & d["ff"].notna()].copy()
d["fp"] = (d["pg"] == "female").astype(float)
d["ff"] = d["ff"].astype(float)
d["year"] = d["dt"].dt.year.astype(str)
d["cell"] = d["investor_uuid"] + "|" + d["year"]
d["us"] = d["country_code"] == "USA"
n_fpff = int(((d["fp"] == 1) & (d["ff"] == 1)).sum())
base_ff = float(d["ff"].mean())
base_fp = float(d["fp"].mean())

# 결과변수: 후속 라운드(36m) · 출구(72m)
r_org = rounds[["org_uuid", "dt"]].sort_values(["org_uuid", "dt"]).copy()
r_org["next_dt"] = r_org.groupby("org_uuid")["dt"].shift(-1)
next_map = r_org.drop_duplicates(["org_uuid", "dt"]).set_index(["org_uuid", "dt"])["next_dt"]
d["next_dt"] = pd.Series(list(zip(d["org_uuid"], d["dt"]))).map(next_map).to_numpy()
d["fon"] = ((pd.to_datetime(d["next_dt"]) - d["dt"]).dt.days <= 365 * 3).fillna(False).astype(float)

acq = CTX.acq.dropna(subset=["acquiree_uuid", "acquired_on"]).copy()
acq["adt"] = pd.to_datetime(acq["acquired_on"], errors="coerce")
first_acq = acq.groupby("acquiree_uuid")["adt"].min()
ip = CTX.ipos.dropna(subset=["org_uuid", "went_public_on"]).copy()
ip["idt"] = pd.to_datetime(ip["went_public_on"], errors="coerce")
first_ipo = ip.groupby("org_uuid")["idt"].min()
exit_dt = pd.concat([first_acq, first_ipo], axis=1).min(axis=1)
d["exit_dt"] = d["org_uuid"].map(exit_dt)
d["exit6"] = ((d["exit_dt"] - d["dt"]).dt.days <= 365 * 6).fillna(False).astype(float)


def fwl(df, y, x, cell="cell", cl="investor_uuid", nb=NB):
    dd = df[[y, x, cell, cl]].reset_index(drop=True)
    yr = dd[y] - dd[y].groupby(dd[cell]).transform("mean")
    xr = dd[x] - dd[x].groupby(dd[cell]).transform("mean")
    yr, xr = yr.to_numpy(), xr.to_numpy()
    sxx = (xr * xr).sum()
    if sxx == 0 or len(dd) < 500:
        return float("nan"), [float("nan")] * 2, int(len(dd))
    beta = float((xr * yr).sum() / sxx)
    grp = {c: g.index.to_numpy() for c, g in dd.groupby(cl)}
    keys = list(grp)
    bs = []
    for _ in range(nb):
        pick = rng.integers(0, len(keys), len(keys))
        rows_ = np.concatenate([grp[keys[i]] for i in pick])
        x2, y2 = xr[rows_], yr[rows_]
        s = (x2 * x2).sum()
        if s:
            bs.append((x2 * y2).sum() / s)
    return beta, qci(bs), int(len(dd))


# H1 동류교배
b_h1, ci_h1, n_h1 = fwl(d, "ff", "fp")
b_h1_us, ci_h1_us, n_us = fwl(d[d["us"]], "ff", "fp")
b_h1_nus, ci_h1_nus, n_nus = fwl(d[~d["us"]], "ff", "fp")


# H2 판별 — gap_ff, gap_mf 와 판별자 (공동 투자사 부트)
def gaps(df, y, nb=NB):
    sub = {s: df[df["ff"] == s][[y, "fp", "cell", "investor_uuid"]].reset_index(drop=True)
           for s in (1.0, 0.0)}
    res, arrs = {}, {}
    for s, dd in sub.items():
        yr = (dd[y] - dd[y].groupby(dd["cell"]).transform("mean")).to_numpy()
        xr = (dd["fp"] - dd["fp"].groupby(dd["cell"]).transform("mean")).to_numpy()
        arrs[s] = (yr, xr, {c: g.index.to_numpy() for c, g in dd.groupby("investor_uuid")})
        sxx = (xr * xr).sum()
        res[s] = float((xr * yr).sum() / sxx) if sxx else float("nan")
    bs = {1.0: [], 0.0: [], "diff": []}
    keys = {s: list(arrs[s][2]) for s in arrs}
    for _ in range(nb):
        cur = {}
        for s in (1.0, 0.0):
            yr, xr, grp = arrs[s]
            ks = keys[s]
            pick = rng.integers(0, len(ks), len(ks))
            rows_ = np.concatenate([grp[ks[i]] for i in pick])
            x2, y2 = xr[rows_], yr[rows_]
            sx = (x2 * x2).sum()
            cur[s] = (x2 * y2).sum() / sx if sx else np.nan
            bs[s].append(cur[s])
        bs["diff"].append(cur[1.0] - cur[0.0])
    return (res[1.0], qci(bs[1.0]), res[0.0], qci(bs[0.0]),
            res[1.0] - res[0.0], qci(bs["diff"]),
            int((df["ff"] == 1).sum()), int((df["ff"] == 0).sum()))


d_exit = d[d["dt"] <= "2017-10-31"]
d_fon = d[d["dt"] <= "2020-10-31"]
gff_e, ciff_e, gmf_e, cimf_e, dj_e, cidj_e, nf_e, nm_e = gaps(d_exit, "exit6")
gff_f, ciff_f, gmf_f, cimf_f, dj_f, cidj_f, nf_f, nm_f = gaps(d_fon, "fon")

k1 = (cov_pg < 0.60) or (cov_ff < 0.40) or (n_fpff < 2000)
k2 = (np.isnan(b_h1)) or (ci_h1[0] <= 0 <= ci_h1[1])
w_e = ciff_e[1] - ciff_e[0] if not np.isnan(ciff_e[0]) else np.inf
w_f = ciff_f[1] - ciff_f[0] if not np.isnan(ciff_f[0]) else np.inf
k3 = (w_e > 0.20) and (w_f > 0.20)
kills = {"K1_coverage": bool(k1), "K2_no_homophily": bool(k2), "K3_adjudication_power": bool(k3)}

screening = (not k2) and (ciff_e[0] > -0.10) and (ciff_f[0] > -0.10)
favoritism = (not k2) and ((ciff_e[1] < 0 and cidj_e[1] < 0) or (ciff_f[1] < 0 and cidj_f[1] < 0))
status = "KILL" if any(kills.values()) else ("GO" if (screening or favoritism) else "PARTIAL")
verdict = (f"동류교배={b_h1:+.4f} {ci_h1} (기저 ff {base_ff:.3f}·fp {base_fp:.3f}; n={n_h1}); "
           f"출구 gap_ff={gff_e:+.4f} {ciff_e} vs gap_mf={gmf_e:+.4f} {cimf_e}, 판별자={dj_e:+.4f} {cidj_e}; "
           f"후속 gap_ff={gff_f:+.4f} {ciff_f}, 판별자={dj_f:+.4f} {cidj_f}; "
           f"US={b_h1_us:+.4f} {ci_h1_us} vs 비US={b_h1_nus:+.4f} {ci_h1_nus}; "
           f"커버리지 파트너성별 {cov_pg:.3f}·창업자성별 {cov_ff:.3f}; "
           + ("선별 방향" if screening else ("호의 방향" if favoritism else "판별 미결")))

emit("I-73", "여성 파트너×여성 창업자 — 호의 vs 선별 판별 (K-3 본추정)", status,
     {"homophily": None if np.isnan(b_h1) else round(b_h1, 4), "homophily_ci": ci_h1, "n_deals": n_h1,
      "base_ff": round(base_ff, 4), "base_fp": round(base_fp, 4), "n_fp_ff": n_fpff,
      "exit_gap_ff": [round(gff_e, 4), ciff_e, nf_e], "exit_gap_mf": [round(gmf_e, 4), cimf_e, nm_e],
      "exit_adjudicator": [round(dj_e, 4), cidj_e],
      "fon_gap_ff": [round(gff_f, 4), ciff_f, nf_f], "fon_gap_mf": [round(gmf_f, 4), cimf_f, nm_f],
      "fon_adjudicator": [round(dj_f, 4), cidj_f],
      "homophily_us": [None if np.isnan(b_h1_us) else round(b_h1_us, 4), ci_h1_us, n_us],
      "homophily_nonus": [None if np.isnan(b_h1_nus) else round(b_h1_nus, 4), ci_h1_nus, n_nus],
      "cov_partner_gender": round(cov_pg, 4), "cov_founder_gender": round(cov_ff, 4),
      "kills": kills},
     prediction="동류교배 +2~+8pp CI 배제; gap_ff ≥ 0(선별 방향), 판별자 −10pp 이하 배제; 두 결과변수 부호 일치",
     verdict=verdict, kill_met=any(kills.values()), n=n_h1,
     extra={"stage": 2, "feeds": "STAGE2_REPORT.md#K-3", "slug": "gender_screening"})
print("done")
